# A LoKI batch: the UI

Every button below also has its handler called directly in the cell that
follows it, so the notebook runs end to end without clicking anything, and the
executed notebook shows the same results a click would.

## Setup

In [ ]:
import tempfile
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plopp as pp
from IPython.display import display

from ess.apps import loki
from ess.apps.batch import (
    TriggerLoop,
    apply,
    backlog,
    batch_table,
    dataset_table,
    reprocess,
    trigger_status,
)
from ess.apps.client import local
from ess.apps.rules import AsOf, Bound, Like, Lookup, LookupEntry, Rule, Selector, Template
from ess.apps.sources import FolderSource
from ess.apps.spec import dataset_ref
from ess.reduce.spec.parameters import QEdges

journal = pd.DataFrame.from_records(
    [
        (60384, 'porous silica', 'transmission'),
        (60385, 'porous silica', 'sample'),
        (60386, 'AgBeh', 'transmission'),
        (60387, 'AgBeh', 'sample'),
        (60388, 'deuterated SDS', 'transmission'),
        (60389, 'deuterated SDS', 'sample'),
        (60392, 'empty beam', 'empty-beam'),
        (60393, 'solvent', 'background'),
        (60394, 'ISIS polymer', 'transmission'),
        (60395, 'ISIS polymer', 'sample'),
    ],
    columns=['run', 'sample', 'role'],
).set_index('run')


In [ ]:
cache = loki.cache()
root = Path(tempfile.mkdtemp(prefix='loki-batch-ui-'))
incoming = root / 'incoming'
incoming.mkdir()


def arrive(*runs):
    """Let runs arrive: link their tutorial files into the folder the source reads."""
    for run in runs:
        (path,) = cache.glob(f'{run}-*.nxs')
        (incoming / path.name).symlink_to(path)


arrive(*journal.index.drop([60394, 60395]))

client = local(
    root / 'store',
    instrument='loki',
    proposal='p1',
    submitter='notebook',
    registry=loki.registry(),
    sources=[
        FolderSource(
            incoming,
            '*.nxs',
            identity=r'(?P<run>\d+)-.*',
            instrument='loki',
            journal=journal.to_dict('index'),
        )
    ],
)


def run_ref(number):
    return dataset_ref(instrument='loki', run=number)


def curves(label, names=None):
    """The I(Q) of every member of a batch, keyed by member or by a name for it."""
    names = names or {}
    return {
        names.get(r.request.member_key, r.request.member_key): client.output(r, 'iofq')
        for r in client.batch(label)
    }


def show(out, *values):
    """Display values inside an Output widget, replacing what was there."""
    out.clear_output()
    with out:
        for value in values:
            if value is not None:
                display(value)


def into(out, fn):
    """A button handler: `fn` never displays; only a click renders its result in `out`."""

    def handler(_=None):
        result = fn()
        show(out, *(result if isinstance(result, tuple) else (result,)))
        return result

    return handler


## Reduce a batch

Fill in the sample and transmission run for each row, add a Q override where
needed, and press Reduce. Background run, empty-beam run, direct beam, and the
beam centre (from the AgBeh run) are set once above the rows.

In [ ]:
center = client.run(loki.BEAM_CENTER, {'sample_run': run_ref(60387)})

shared = {
    'background_run': run_ref(60393),
    'background_transmission_run': run_ref(60392),
    'empty_beam_run': run_ref(60392),
    'direct_beam': dataset_ref(path=cache / 'direct-beam-loki-all-pixels.h5'),
    'beam_center': center.ref(),
    'q': QEdges(start=0.01, stop=0.3, num_bins=100),
}
template = Template(
    name='loki-iofq-larmor',
    spec=loki.IOFQ.id,
    params=shared,
    blanks=('sample_run', 'sample_transmission_run'),
    dataset_field='sample_run',
)

shared_settings = widgets.HTML(
    '<b>Shared settings</b> &nbsp; '
    f'background run: {shared["background_run"]} &nbsp; '
    f'empty-beam run: {shared["empty_beam_run"]} &nbsp; '
    f'direct beam: {shared["direct_beam"]} &nbsp; '
    f'beam centre: {shared["beam_center"]}'
)


In [ ]:
def options_for(role):
    """Dropdown options for a journal role: run number and sample name to a ref."""
    return [
        (f'{d.run} {d.fields.get("sample", "")}', d.ref)
        for d in sorted(client.datasets(), key=lambda d: d.run or 0)
        if d.fields.get('role') == role
    ]


def default_ref(options, name):
    """The option of `options` (label, ref pairs) whose label names this sample."""
    return next(ref for label, ref in options if name in label)


def make_row(name):
    sample_opts, transmission_opts = options_for('sample'), options_for('transmission')
    return {
        'name': name,
        'sample_run': widgets.Dropdown(
            options=sample_opts, value=default_ref(sample_opts, name)
        ),
        'transmission_run': widgets.Dropdown(
            options=transmission_opts, value=default_ref(transmission_opts, name)
        ),
        'override_q': widgets.Checkbox(value=False, description='override Q', indent=False),
        'q_start': widgets.FloatText(value=0.01, description='start'),
        'q_stop': widgets.FloatText(value=0.3, description='stop'),
        'q_bins': widgets.IntText(value=100, description='bins'),
    }


def row_box(row):
    return widgets.HBox(
        [
            widgets.Label(row['name'], layout=widgets.Layout(width='120px')),
            row['sample_run'],
            row['transmission_run'],
            row['override_q'],
            row['q_start'],
            row['q_stop'],
            row['q_bins'],
        ]
    )


rows = {name: make_row(name) for name in ['porous silica', 'AgBeh', 'deuterated SDS']}
batch_out = widgets.Output()
reduce_button = widgets.Button(description='Reduce', button_style='primary')


def pinned_table():
    """The batch form as the pinned frame `apply` takes: one row per member."""
    table = {}
    for name, row in rows.items():
        entry = {
            'sample_run': row['sample_run'].value,
            'sample_transmission_run': row['transmission_run'].value,
        }
        if row['override_q'].value:
            entry['q'] = QEdges(
                start=row['q_start'].value,
                stop=row['q_stop'].value,
                num_bins=row['q_bins'].value,
            )
        table[name] = entry
    return pd.DataFrame(table).T


def reduce_batch(_=None):
    """Validate and submit the batch form: the batch table and plot, or the errors."""
    group = apply(client, template, pinned=pinned_table(), label='samples')
    reports = {key: client.validate(request) for key, request in group.items()}
    if any(not report.ok for report in reports.values()):
        return pd.DataFrame({k: r.model_dump() for k, r in reports.items()}).T, None
    client.submit_group(group)
    return batch_table(client, 'samples'), pp.plot(curves('samples'), norm='log')


reduce_button.on_click(into(batch_out, reduce_batch))
display(widgets.VBox([shared_settings, *[row_box(r) for r in rows.values()], reduce_button, batch_out]))


In [ ]:
batch_table_result, batch_plot = reduce_batch()
batch_table_result

In [ ]:
batch_plot

### Correct one member

Deuterated SDS wants a lower Q edge. Check its Q override, set the start to
0.005, and press Reduce again.

In [ ]:
rows['deuterated SDS']['override_q'].value = True
rows['deuterated SDS']['q_start'].value = 0.005

In [ ]:
batch_table_result, batch_plot = reduce_batch()
batch_table_result

## Automatic reduction

Name the rule, pick which journal role it selects, and the run number below
which it does not fire on its own. The lookup fills the transmission run with
the nearest transmission run measured before the member. Press Enable.

In [ ]:
rule_name_w = widgets.Text(value='iofq-auto', description='rule name')
role_w = widgets.Dropdown(
    options=sorted({d.fields['role'] for d in client.datasets()}),
    value='sample',
    description='selector role',
)
bound_w = widgets.IntText(value=60393, description='bound run')
enable_button = widgets.Button(description='Enable', button_style='primary')
status_out = widgets.Output()

rule = None
loop = None


def with_samples(table):
    """A rule's batch table with the sample name of each member."""
    return table.join(dataset_table(client)['sample']).set_index('sample', append=True)


def trigger_table():
    """The trigger status of every dataset the source knows, for the active rule."""
    return dataset_table(client).join(
        pd.DataFrame(
            [
                {'dataset': str(d.ref), **trigger_status(client, rule, d).model_dump()}
                for d in client.datasets()
            ]
        ).set_index('dataset')
    )


def enable_rule(_=None):
    """Create the rule and its trigger loop; the trigger status of every dataset."""
    global rule, loop
    nearest_transmission = LookupEntry(
        name='nearest-transmission',
        fills={
            'sample_transmission_run': AsOf(match={'role': Like(pattern='transmission')})
        },
    )
    rule = Rule(
        name=rule_name_w.value,
        template=template,
        lookup=Lookup(name='transmission', entries=(nearest_transmission,)),
        selector=Selector(
            match={'role': Like(pattern=role_w.value)}, after=Bound(run=bound_w.value)
        ),
    )
    loop = TriggerLoop(client, rule)
    return trigger_table()


enable_button.on_click(into(status_out, enable_rule))
display(
    widgets.VBox(
        [
            widgets.HBox([rule_name_w, role_w, bound_w]),
            widgets.Label('lookup: nearest transmission before the member'),
            enable_button,
            status_out,
        ]
    )
)


In [ ]:
enable_rule()

### Let the polymer arrive

The polymer and its transmission run were held back. Let them arrive, then
press Check now.

In [ ]:
arrive_button = widgets.Button(description='Arrive 60394, 60395')


def arrive_polymer(_=None):
    arrive(60394, 60395)
    return trigger_table()


arrive_button.on_click(into(status_out, arrive_polymer))
display(arrive_button)

In [ ]:
arrive_polymer()

In [ ]:
check_button = widgets.Button(description='Check now')
batch_status_out = widgets.Output()


def check_now(_=None):
    """Run the trigger loop once: what fired, the status table, the rule's batch."""
    fired = loop.run_once()
    return fired, trigger_table(), with_samples(batch_table(client, rule))


def on_check_clicked(_=None):
    _fired, status, batch = check_now()
    show(status_out, status)
    show(batch_status_out, batch)


check_button.on_click(on_check_clicked)
display(widgets.VBox([check_button, batch_status_out]))

In [ ]:
fired, status_table, rule_batch_table = check_now()
[(r.request.member_key, r.status.value) for r in fired]

In [ ]:
status_table

In [ ]:
rule_batch_table

Pressing Check now again fires nothing: the loop keeps no memory of what it already fired on.

In [ ]:
fired_again, _, _ = check_now()
len(fired_again)

## Backlog

The three samples measured before the bound were never reduced under this
rule. Press Reduce backlog to submit them.

In [ ]:
backlog_button = widgets.Button(description='Reduce backlog', button_style='primary')


def reduce_backlog(_=None):
    client.submit_group(backlog(client, rule))
    return with_samples(batch_table(client, rule))


backlog_button.on_click(into(batch_status_out, reduce_backlog))
display(widgets.VBox([backlog_button, batch_status_out]))

In [ ]:
reduce_backlog()

## Correct a member of the rule's batch

Pick a member, pin a new Q start, and press Reduce member.

In [ ]:
def rule_members():
    """The rule's current members, as (sample name, dataset) pairs for the dropdown."""
    keys = {str(r.request.member_key) for r in client.batch(rule.name)}
    return [(d.fields.get('sample', str(d.ref)), d) for d in client.datasets() if str(d.ref) in keys]


polymer = next(d for d in client.datasets() if d.run == 60395)
member_w = widgets.Dropdown(options=rule_members(), value=polymer, description='member')
q_start_w = widgets.FloatText(value=0.005, description='start')
q_stop_w = widgets.FloatText(value=0.3, description='stop')
q_bins_w = widgets.IntText(value=100, description='bins')
reduce_member_button = widgets.Button(description='Reduce member', button_style='primary')


def reduce_member(_=None):
    """Pin a Q override for one member of the rule's batch, and reduce it."""
    member = member_w.value
    q = QEdges(start=q_start_w.value, stop=q_stop_w.value, num_bins=q_bins_w.value)
    client.submit_group(apply(client, rule, [member], {str(member.ref): {'q': q}}))
    return with_samples(batch_table(client, rule))


reduce_member_button.on_click(into(batch_status_out, reduce_member))
display(
    widgets.VBox(
        [
            widgets.HBox([member_w, q_start_w, q_stop_w, q_bins_w]),
            reduce_member_button,
            batch_status_out,
        ]
    )
)

In [ ]:
reduce_member()

## Revise the defaults

Move the template's default Q binning to 200 bins, save it as a new rule
version, and reprocess the members still on an older version.

In [ ]:
bins_w = widgets.IntText(value=200, description='default Q bins')
save_button = widgets.Button(description='Save as new version')
reprocess_button = widgets.Button(description='Reprocess', button_style='primary')
reprocess_out = widgets.Output()

rule_v2 = None


def save_new_version(_=None):
    global rule_v2
    template_v2 = template.revise(q=QEdges(start=0.01, stop=0.3, num_bins=bins_w.value))
    rule_v2 = rule.revise(template=template_v2)
    return rule_v2.id


def reprocess_rule(_=None):
    client.submit_group(reprocess(client, rule_v2))
    return with_samples(batch_table(client, rule_v2))


save_button.on_click(into(reprocess_out, save_new_version))
reprocess_button.on_click(into(reprocess_out, reprocess_rule))
display(widgets.VBox([bins_w, widgets.HBox([save_button, reprocess_button]), reprocess_out]))

In [ ]:
save_new_version()

In [ ]:
reprocess_rule()

### The polymer's history

Every record made for the polymer under this rule: the loop's, the
correction, and the reprocess. The last row is the one the batch table shows.

In [ ]:
pd.DataFrame(
    [
        {
            'record': r.id,
            'rule': r.request.origin.rule,
            'pinned': list(r.request.origin.pinned),
            'q_start': r.resolved_params['q']['start'],
            'q_bins': r.resolved_params['q']['num_bins'],
        }
        for r in client.records(label=rule.name, member_key=str(polymer.ref))
    ]
)

In [ ]:
names = {str(run_ref(n)): s for n, s in journal['sample'].items()}
pp.plot(curves(rule.name, names), norm='log')